In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("barkataliarbab/udacity-self-driving-car-obstacles-dataset")

print("Path to dataset files:", path)

100%|██████████| 1.08G/1.08G [01:26<00:00, 13.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/barkataliarbab/udacity-self-driving-car-obstacles-dataset/versions/1


In [3]:
import os
os.listdir(path)

['export', 'data.yaml', 'README.dataset.txt', 'README.roboflow.txt']

In [4]:
base_path=os.listdir(os.path.join(path,'export'))
base_path

['images', 'labels']

In [5]:
label_file=os.listdir(os.path.join(path,'export/labels'))[0]
with open(os.path.join(path,'export/labels',label_file),'r') as f:
    print(f.read())

1 0.8447265625 0.4697265625 0.078125 0.0703125
1 0.8486328125 0.4697265625 0.072265625 0.0498046875


In [6]:
# pip install ultralytics

In [7]:
# image_file=os.listdir(os.path.join(path,'export/images'))[0]
# with open(os.path.join(path,'export/images',label_file),'r') as f:
#     print(f.read())

In [8]:
import shutil
import os

# Define paths
src_path = path
dst_path = "/kaggle/working/dataset"

# Copy the dataset to the writable directory
shutil.copytree(src_path, dst_path, dirs_exist_ok=True)

print("Copied to:", dst_path)

Copied to: /kaggle/working/dataset


In [9]:
path=dst_path
base_path = os.path.join(path, "export")
base_path

'/kaggle/working/dataset/export'

In [10]:
images_path = os.path.join(base_path, "images")
labels_path = os.path.join(base_path, "labels")

train_img = os.path.join(base_path, "train/images")
train_lbl = os.path.join(base_path, "train/labels")

val_img = os.path.join(base_path, "valid/images")
val_lbl = os.path.join(base_path, "valid/labels")

os.makedirs(train_img, exist_ok=True)
os.makedirs(train_lbl, exist_ok=True)
os.makedirs(val_img, exist_ok=True)
os.makedirs(val_lbl, exist_ok=True)

print("Folders created ✅")

Folders created ✅


In [11]:
from sklearn.model_selection import train_test_split
import os

images = os.listdir(images_path)

train_files, val_files = train_test_split(
    images,
    test_size=0.2,   # 20% validation
    random_state=42, # reproducibility
    shuffle=True
)

print("Train:", len(train_files), "Val:", len(val_files))

Train: 23840 Val: 5960


In [12]:
import shutil

def copy_files(file_list, img_dest, lbl_dest):
    for file in file_list:
        img_src = os.path.join(images_path, file)

        name = os.path.splitext(file)[0]
        label_file = name + ".txt"
        lbl_src = os.path.join(labels_path, label_file)

        # copy image
        if os.path.exists(img_src):
            shutil.copy(img_src, os.path.join(img_dest, file))

        # copy label
        if os.path.exists(lbl_src):
            shutil.copy(lbl_src, os.path.join(lbl_dest, label_file))

copy_files(train_files, train_img, train_lbl)
copy_files(val_files, val_img, val_lbl)

print("Dataset split complete ✅")

Dataset split complete ✅


In [13]:
import yaml

yaml_path = os.path.join(path, "data.yaml")

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(base_path, "train/images")
data['val'] = os.path.join(base_path, "valid/images")

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("YAML updated ✅")

YAML updated ✅


In [14]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.0 MB/s eta 0:00:00


In [16]:
from ultralytics import YOLO
import os

model = YOLO('yolov5n.pt')  # or 'yolov8l.pt' - medium/large
model.train(
    data=path,
    epochs=15,           # 3-5x more epochs
    imgsz=640,            # higher resolution
    batch=16,             # adjust based on GPU
    patience=20,          # early stopping
    augment=True,         # enable augmentation
    mosaic=1.0,           # keep mosaic augmentation
    mixup=0.2,            # add mixup
    copy_paste=0.3        # for instance segmentation style
)

PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.

Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, l

KeyboardInterrupt: 

In [ ]:
results = model.predict(source=val_img, show=True)

In [ ]:
#save the best model
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train5/weights/best.pt")
model.export(format="onnx")